In [ ]:
import os
import glob
import torch
from datasets import load_dataset

# import numpy
# import numpy._core.multiarray
# torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"

# 
# -----------------------------------------------------------------------
OUTPUT_DIR = "/teamspace/studios/this_studio/fixmaster-lora-v1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the Heavy Dataset from Hugging Face
print("Connecting to Hugging Face Data Stream...")
dataset = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/madhavpnair/CommitPackFT_proc2/resolve/main/fixmaster_training_data_clean.jsonl",
    split="train"
)

def format_dataset(example):
    """
    Maps the JSONL columns into the strict FixMaster format.
    Assumes your JSONL has "prompt" (the error) and "patch" (the fix) keys.
    """
    text = f"""<|im_start|>system
You are FixMaster, an autonomous security remediation agent. Given an error log and buggy code, output ONLY the unified Git patch. Do not output conversational text.<|im_end|>
<|im_start|>user
{example['prompt']}<|im_end|>
<|im_start|>assistant
```diff
{example['patch']}
```<|im_end|>"""
    return {"text": text}

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("Configuring 4-bit quantization for T4 GPU...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading base model into GPU...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,   # force fp16 to match compute_dtype, avoids bf16/GradScaler crash
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

print("Attaching LoRA adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)   # attach LoRA manually, not passed to trainer

# Apply the formatting function to the dataset, replacing prompt/patch columns with "text"
formatted_dataset = dataset.map(format_dataset, remove_columns=dataset.column_names)

print("Initializing Heavy SFT Trainer...")
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",

    max_length=2048,             # renamed from max_seq_length (TRL 1.9.0 API)

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,

    num_train_epochs=1,

    optim="paged_adamw_8bit",
    fp16=False,                  # GradScaler disabled entirely (was crashing on bf16 tensors)
    bf16=False,

   
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,

    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    # peft_config not passed here -- model is already PEFT-wrapped above
    args=sft_config,
)

# --------- use saved checkpoints ------------------
checkpoints = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
resume_path = None
if checkpoints:
    checkpoints.sort(key=lambda p: int(p.split("-")[-1]))
    resume_path = checkpoints[-1]
    print(f"Found existing checkpoint: {resume_path}. Resuming from it.")
else:
    print("No existing checkpoint found. Starting fresh.")

print("Beginning Heavy Fine-Tuning... (This may take several hours!)")
trainer.train(resume_from_checkpoint=resume_path)

print("Saving final adapters...")
trainer.model.save_pretrained(f"{OUTPUT_DIR}-final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}-final")
print("Training Complete!")